# 숙제 · 좋은 그림 그리기 (15 ~ 17강)

화학데이터사이언스 2026-2

**과제**: 그림을 **3 장 이상** 그려 온다.

| | 몇 장 | 무엇을 |
|---|---|---|
| ① 목표 그림 재현 + 개선 | 2 장 이상 | 카드 A ~ G 중에서 고른다. 목표 그림을 **처음부터 코드로** 다시 그리고, **더 좋게** 고친다 |
| ② 자유 그림 | 1 장 이상 | 목표 그림 없음. 질문 메뉴(또는 스스로 정한 질문)와 데이터로 형식부터 직접 정한다 |

> ⚠️ **목표 그림은 완벽하지 않다.** 그림마다 15 강에서 말한 '나쁜 그림'의 요소가 **몇 군데 숨어 있다.**
> 똑같이 따라 그리기만 하면 점수가 낮다. 무엇이 문제인지 찾아서 고친 만큼 점수가 오른다.

**AI 사용**: ChatGPT · Gemini · Claude 등 **써도 된다.** 대신 맨 아래 'AI 사용 기록' 칸에 무엇을 물었는지 적는다.
AI 는 목표 그림을 그대로 따라 그리는 데는 빠르지만, 무엇이 잘못됐는지는 **여러분이** 판단해야 한다.

---

### ▶ Colab 에서 시작하기
1. 링크로 열었다면 먼저 `파일 → Drive에 사본 저장` (그래야 내 코드가 저장된다)
2. "GitHub 에서 로드되었습니다" 경고가 뜨면 `무시하고 계속`
3. 셀은 위에서부터 `Shift + Enter` 로 하나씩 실행
4. 그림 안 글자는 **영어**로 쓴다 (Colab 기본 글꼴은 한글이 □□ 로 깨진다)

## 0. 준비 (실행만)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BASE = "https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/data/"

## 1. 데이터 둘러보기

- 아래 셀은 **실행만** 하면 된다. 데이터는 GitHub 에서 바로 읽으니 파일을 올릴 필요가 없다.
- `df.head()` 로 열 이름과 값의 모양을 먼저 본다. 그림을 그리기 전에 **어떤 열을 x, y 로 쓸지** 정한다.

| 파일 | 내용 | 출처 |
|---|---|---|
| `titration.csv` | 0.100 M 산 25.00 mL 를 0.100 M NaOH 로 적정할 때 pH (HCl · 아세트산 · 인산) | 문헌 pKa 로 전하 균형식을 풀어 계산 |
| `speciation.csv` | pH 0–14 에서 화학종 분율 (탄산 · 인산 · 시트르산 · 글라이신), 긴 표 | 문헌 pKa 로 계산 |
| `cathode_composition.csv` | 리튬 이온 전지 양극재 9 종의 원소 질량 백분율 · 몰질량 · 이론 용량 | 화학식으로 계산 (pymatgen) |
| `tio2_xrd.csv` | TiO₂ 아나타제 · 루타일의 XRD 패턴 (Cu Kα) | 문헌 결정 구조로 계산 (pymatgen) + 잡음. 측정값 아님 |
| `elements.csv` | 원소 86 개의 주기 · 족 · 블록 · 전기음성도 · 원자 반지름 · 이온화에너지 · 녹는점 · 밀도 | pymatgen 원소 자료 |
| `liquids.csv` | 액체 6 종(물 · 메탄올 · 에탄올 · 벤젠 · 톨루엔 · 헥세인)의 온도별 증기압 · 밀도 · 점도 | CoolProp 물성 상태식으로 계산 (1 atm) |

In [ ]:
titration = pd.read_csv(BASE + "titration.csv")
print(titration.shape)
titration.head()

In [ ]:
speciation = pd.read_csv(BASE + "speciation.csv")
print(speciation.shape)
speciation.head()

In [ ]:
cathode_composition = pd.read_csv(BASE + "cathode_composition.csv")
print(cathode_composition.shape)
cathode_composition.head()

In [ ]:
tio2_xrd = pd.read_csv(BASE + "tio2_xrd.csv")
print(tio2_xrd.shape)
tio2_xrd.head()

In [ ]:
elements = pd.read_csv(BASE + "elements.csv")
print(elements.shape)
elements.head()

In [ ]:
liquids = pd.read_csv(BASE + "liquids.csv")
print(liquids.shape)
liquids.head()

**표 모양 팁**
- `speciation.csv` 는 **긴 표**다 (한 줄 = 산 · 화학종 · pH 하나). 누적 그림을 그리려면 `pivot` 으로 넓은 표로 바꾼다 (17강 heatmap 에서 한 것과 같다).
- `liquids.csv` 는 액체 6 종이 한 표에 있다. `liquids[liquids.liquid == "water"]` 처럼 골라 쓴다.

## 2. 목표 그림 카드 (A ~ G)

- 2 장 이상 고른다. 같은 기법(예: 다중 패널 C 와 G)만 고르기보다 **서로 다른 기법**을 고르면 더 많이 배운다.
- 목표 그림을 보며 스스로 묻는다: **축에 단위가 있나? 글자는 읽히나? 색이 뜻하는 게 보이나? 패널끼리 비교가 되나? 질문의 답이 그림에서 보이나?**

### 카드 A · 적정 곡선

- **질문**: 아세트산은 왜 pH 변화를 버티나? — 강산 · 약산 · 다양성자산의 적정 곡선
- **데이터**: `titration.csv`  ·  **기법**: fill_between (where=)  ·  **강의 힌트**: 17강 앞부분 · 일반 그림 템플릿, fill_between

![카드 A 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/A_target.png)

### 카드 B · 양극재 조성

- **질문**: 양극재에 코발트는 얼마나 들어 있나? — LCO 에서 NMC, LFP 까지 원소 질량 백분율
- **데이터**: `cathode_composition.csv`  ·  **기법**: stacked bar  ·  **강의 힌트**: 수업 코발트 생산 점유율 실습 (누적 막대) · 5강 molar_mass

![카드 B 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/B_target.png)

### 카드 C · TiO₂ 다형체 XRD

- **질문**: 같은 TiO₂, 다른 구조 — 아나타제와 루타일의 XRD 패턴은 어떻게 다른가?
- **데이터**: `tio2_xrd.csv`  ·  **기법**: 다중 패널 (GridSpec, x축 공유)  ·  **강의 힌트**: 17강 다중 패널 (GridSpec, 약 15:40 ~ 23:30)

![카드 C 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/C_target.png)

### 카드 D · 주기율표 heatmap

- **질문**: 첫 이온화에너지는 주기율표에서 어떻게 변하나? — 원자번호 순서와 주기율표 배치
- **데이터**: `elements.csv`  ·  **기법**: heatmap + 곁 패널  ·  **강의 힌트**: 17강 heatmap (약 23:30 ~ 36:00) · 표를 격자로 바꾸기

![카드 D 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/D_target.png)

### 카드 E · 물의 밀도와 증기압

- **질문**: 물은 몇 도에서 가장 무겁고, 언제 끓나? — 밀도와 증기압을 한 그림에
- **데이터**: `liquids.csv`  ·  **기법**: y축 두 개 (twinx)  ·  **강의 힌트**: 17강 multiple axes (twinx)

![카드 E 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/E_target.png)

### 카드 F · 원소 성질 산점도

- **질문**: 전기음성도가 큰 원자는 전자를 더 꽉 붙잡나? — 전기음성도와 첫 이온화에너지
- **데이터**: `elements.csv`  ·  **기법**: 산점도 · 그룹별 색 · 선택적 라벨  ·  **강의 힌트**: 16강 기본 그림 (scatter) · groupby

![카드 F 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/F_target.png)

### 카드 G · 화학종 분포 (2×2)

- **질문**: pH 마다 어떤 화학종이 가장 많은가? — 탄산 · 인산 · 시트르산 · 글라이신
- **데이터**: `speciation.csv`  ·  **기법**: 2×2 다중 패널 + stackplot  ·  **강의 힌트**: 17강 다중 패널 · 수업 누적 그림(stackplot) · 17강 heatmap 의 pivot

![카드 G 목표 그림](https://raw.githubusercontent.com/gwanyeong/my_projects/master/chem-data-science/hw_plotting/targets/G_target.png)

## 3. 자유 그림 — 질문 메뉴 (목표 그림 없음)

하나를 고르거나, 이 데이터로 **스스로 질문을 만든다.** 어떤 그림 형식이 질문에 맞는지부터 정한다.

| 데이터 | 질문 |
|---|---|
| liquids.csv | 어떤 액체가 가장 잘 날아가나? 증기압으로 휘발성을 비교하고, ln P 대 1/T 기울기로 증발 엔탈피 ΔH$_{vap}$ 를 어림해 보자 (Clausius–Clapeyron). |
| liquids.csv | 점도는 온도에 얼마나 민감한가? 액체마다 다른가? |
| cathode_composition.csv | 코발트를 줄이면 이론 용량을 잃나? 몰질량 · Co 함량 · 이론 용량의 관계. |
| elements.csv | 원자 반지름(또는 녹는점 · 밀도)은 주기율표에서 어떻게 변하나? |
| tio2_xrd.csv | 아나타제 70 % + 루타일 30 % 혼합 시료의 패턴은 어떻게 보일까? 두 패턴을 섞어 그리고, 어떤 봉우리로 루타일을 알아볼지 표시해 보자. |
| speciation.csv · titration.csv | 혈액 pH 7.4 에서 인산과 탄산은 어떤 꼴로 있나? 또는 적정 곡선의 기울기(dpH/dV)로 당량점을 찾아보자. |
| 메뉴 밖 데이터 | NIST WebBook 스펙트럼, PubChem 성질, 실험 수업에서 직접 얻은 데이터 등. 출처를 밝히면 가산점. |

---
## 나의 그림 1 · 카드 ___

In [ ]:
# 그림 1 코드를 여기에 (데이터 읽기부터 plt.show() 까지 이 셀 하나로 그려지게)

**그림 1 · 목표 그림에서 찾은 문제와 고친 방법** (찾은 만큼 줄을 늘린다)

| # | 목표 그림의 문제 | 어떻게 고쳤나 |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

**이 그림이 말하는 것 (화학으로 한 줄)**:

---
## 나의 그림 2 · 카드 ___

In [ ]:
# 그림 2 코드를 여기에 (데이터 읽기부터 plt.show() 까지 이 셀 하나로 그려지게)

**그림 2 · 목표 그림에서 찾은 문제와 고친 방법** (찾은 만큼 줄을 늘린다)

| # | 목표 그림의 문제 | 어떻게 고쳤나 |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

**이 그림이 말하는 것 (화학으로 한 줄)**:

---
## 나의 그림 3 · 자유 (질문: ________________)

In [ ]:
# 그림 3 코드를 여기에 (데이터 읽기부터 plt.show() 까지 이 셀 하나로 그려지게)

**그림 3 · 설명**

- 왜 이 형식(선 · 막대 · 산점도 · 패널 …)을 골랐나:
- 다듬은 점 (최소 3 가지):
- 이 그림이 말하는 것 (화학으로 한 줄):

---
## 더 그리고 싶다면 (가산점)
아래에 셀을 추가해 4 장, 5 장 … 그려도 된다. 같은 형식을 추가하는 셀을 위 칸에서 복사해 쓴다.

---
## AI 사용 기록

- 사용한 도구:
- 주로 무엇을 물었나 (질문 요지 2 ~ 3 줄):
- AI 가 준 코드에서 **내가 직접 고친 것**:

## 제출 전 확인
- [ ] `런타임 → 다시 시작 및 모두 실행` 으로 처음부터 에러 없이 그림이 다 나온다
- [ ] 그림 3 장 이상 (목표 2 + 자유 1)
- [ ] 그림마다 '고친 점' · '말하는 것' 을 썼다
- [ ] AI 사용 기록을 썼다
- [ ] 파일 이름: `HW_plotting_학번_이름.ipynb`